# Train an XGBoost built-in algorithm and compare Bayesian vs random search on a fixed 10-trial HPO budget

The product team's lead engineer has been running random-search HPO on every model for two years. The new senior on the team keeps saying Bayesian is strictly better. You have one session to settle it on this team's actual dataset with this team's actual 10-trial budget. The answer might be that Bayesian wins by 2 AUC points. The answer might be that on 10 trials with a small search space, random is within noise of Bayesian. Either result is fine; the engineer wants the receipt.

Five artifacts ship in this lab:

- A 5000-row tabular binary-classification CSV (loan default prediction) with 12 features, split 70/15/15 into train, validation, and test prefixes in S3 in the XGBoost built-in format (CSV without header, label in column 0).
- A baseline XGBoost training job on `ml.m5.xlarge` with the validation AUC captured from `FinalMetricDataList`.
- Architecture A: a SageMaker Automatic Model Tuning job with `Strategy="Bayesian"`, `MaxNumberOfTrainingJobs=10`, `MaxParallelTrainingJobs=2`, tuning `max_depth`, `eta`, `subsample`, and `min_child_weight`.
- Architecture B: the same HPO job with `Strategy="Random"` and the same budget and ranges.
- A side-by-side summary: best objective per strategy, the trial index at which the best landed (steps-to-best), and the per-trial AUC trajectories.

**Time.** Plan for about 65 minutes hands-on. Each HPO trial runs roughly 5 minutes on ml.m5.xlarge; with 2 trials in parallel, the 10-trial sweep wall-clocks around 25 minutes. Two sweeps total roughly 50 minutes of wall time; the baseline adds 5 more. Budget 120 minutes if a hyperparameter range fights back.

**Cost.** This lab costs about eight cents if everything lands on the first try. SageMaker training on ml.m5.xlarge is $0.230 per hour; one baseline at 5 minutes is about two cents, plus 20 HPO trials at 5 minutes each comes to about eight cents at $0.230 per hour. A realistic session with one config retry lands around $0.08 to $0.15. The cleanup cell stops any in-flight tuning job and tears the S3, IAM, and tagged artifacts down so your bill stops the moment you finish.

This lab maps to AWS MLA-C01 Domain 2 (ML Model Development, 26%) on HPO strategies (random, grid, Bayesian, Hyperband) and SageMaker Automatic Model Tuning. The reflection MCQ surfaces the constraint-driven comparison: which strategy wins under which scenario.

**Compare-it lab.** Two HPO architectures run side by side against the same dataset and budget. Checkpoints validate that BOTH sweeps completed; the comparative reasoning (Bayesian vs random for a fixed trial budget) lives in the reflection MCQs.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the SageMaker Python SDK for image URI lookups.
# Both are pinned to specific versions per LAB-CREATION-BLUEPRINT.md.

!pip install --quiet labrun-checks==0.6.0 sagemaker==2.232.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern; the -bayes and -random suffixes
# disambiguate the two HPO architectures per blueprint Section 21.

import atexit
import csv
import getpass
import io
import json
import random
import sys
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "xgboost-built-in-training-and-hpo"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names. Bucket and ARNs are account-suffixed and resolved after STS returns.
TRAINING_ROLE_NAME = f"labrun-{LAB_ID}-role"
INLINE_POLICY_NAME = "labrun-mla-lab05-training-policy"
BASELINE_JOB = f"labrun-{LAB_ID}-baseline"
BAYES_JOB = f"labrun-{LAB_ID}-bayes"
RANDOM_JOB = f"labrun-{LAB_ID}-random"

BUCKET_NAME = None
ACCOUNT_ID = None
TRAINING_ROLE_ARN = None
XGBOOST_IMAGE_URI = None

# Captured state used by Checkpoint 5 (the Compare-it metric capture).
BASELINE_AUC = None
BAYES_BEST = None
BAYES_STEPS_TO_BEST = None
RANDOM_BEST = None
RANDOM_STEPS_TO_BEST = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names. This cell must succeed before the manifest
# cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 2 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
TRAINING_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{TRAINING_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Training role ARN: {TRAINING_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# Manifest is module-level and in reverse-creation order.
#
# labrun-checks 0.6.0 does not have adapters for sagemaker_training_job or
# sagemaker_tuning_job. The cleanup cell stops any in-flight job manually
# BEFORE run_cleanup walks the manifest. Completed training and tuning jobs
# cannot be deleted via API; they auto-expire and bill nothing once Completed.
# The manifest below contains only adapter-supported types (iam_role,
# s3_bucket). The orphan scan still picks up any tagged SageMaker resource
# via Resource Groups Tagging API.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=TRAINING_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {TRAINING_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Generate the 5000-row training set, split 70/15/15, upload in XGBoost CSV format, stand up the role

The XGBoost built-in algorithm requires CSV without a header row and the label in column 0. Get either of those wrong and `create_training_job` fails with `ClientError: Failed to parse CSV input`. The data generator emits the right shape; you upload the three splits and provision the SageMaker training role.

Build it in this order:

1. Create the S3 bucket `labrun-xgboost-built-in-training-and-hpo-{your-account-id}` and tag it `labrun:lab-slug=xgboost-built-in-training-and-hpo`.
2. Generate a deterministic 5000-row CSV with the helper below (12 features, binary label, label in column 0). The split is roughly 3500 train, 750 validation, 750 test.
3. Upload `train.csv`, `validation.csv`, `test.csv` to the `input/` prefix.
4. Create the IAM role `labrun-xgboost-built-in-training-and-hpo-role` with `sagemaker.amazonaws.com` trust and an inline policy granting S3 read/write on the bucket plus `s3:ListBucket`.

Why the 12 features and the 70/15/15 split? The Compare-it lab needs enough rows that each trial has a meaningful AUC signal but few enough that 10 trials at 5 minutes each lands inside a session window. The 12 features map to the four hyperparameter ranges the HPO sweep tunes (`max_depth`, `eta`, `subsample`, `min_child_weight`), giving Bayesian and Random a non-trivial search surface.

In [ ]:
# NBVAL_SKIP
# Task 1: Generate the deterministic CSV, split 70/15/15, upload, create role.

import sagemaker

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the bucket with s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects CreateBucketConfiguration; other regions require it.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")


def generate_xgboost_rows(n_rows: int) -> list:
    rng = random.Random(42)
    rows = []
    for _ in range(n_rows):
        features = [round(rng.gauss(0, 1), 4) for _ in range(12)]
        logit = (
            0.6 * features[0]
            - 0.4 * features[1]
            + 0.3 * features[2]
            - 0.2 * features[3]
            + 0.1 * features[4]
            + rng.gauss(0, 0.4)
        )
        label = 1 if logit > 0 else 0
        rows.append([label] + features)
    return rows


def rows_to_csv_bytes(rows: list) -> bytes:
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerows(rows)
    return buf.getvalue().encode("utf-8")


all_rows = generate_xgboost_rows(5000)
random.Random(42).shuffle(all_rows)
train_rows = all_rows[:3500]
val_rows = all_rows[3500:4250]
test_rows = all_rows[4250:]

train_bytes = rows_to_csv_bytes(train_rows)
val_bytes = rows_to_csv_bytes(val_rows)
test_bytes = rows_to_csv_bytes(test_rows)

# YOUR CODE: Upload train_bytes to input/train.csv, val_bytes to
# input/validation.csv, test_bytes to input/test.csv using s3.put_object().
print(f"Uploaded s3://{BUCKET_NAME}/input/train.csv ({len(train_rows)} rows)")
print(f"Uploaded s3://{BUCKET_NAME}/input/validation.csv ({len(val_rows)} rows)")
print(f"Uploaded s3://{BUCKET_NAME}/input/test.csv ({len(test_rows)} rows)")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    # YOUR CODE: Create the role using iam.create_role() with
    # RoleName=TRAINING_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
    # Description="labrun mla lab 05 training role", Tags=[{"Key": LAB_TAG_KEY,
    # "Value": LAB_TAG_VALUE}].
    print(f"Created role: {TRAINING_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {TRAINING_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BucketRW",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Sid": "BucketList",
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
    ],
}

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=TRAINING_ROLE_NAME, PolicyName=INLINE_POLICY_NAME,
# PolicyDocument=json.dumps(inline_policy).
print(f"Attached inline policy {INLINE_POLICY_NAME} to {TRAINING_ROLE_NAME}")

print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

XGBOOST_IMAGE_URI = sagemaker.image_uris.retrieve(
    framework="xgboost", region=REGION, version="1.5-1"
)
print(f"XGBoost image URI: {XGBOOST_IMAGE_URI}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: three splits are in S3, no header row, label in column 0,
# row counts approximate the 70/15/15 split of 5000 rows.

def checkpoint_1(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        expectations = {
            "input/train.csv": (3000, 4000),
            "input/validation.csv": (600, 900),
            "input/test.csv": (600, 900),
        }
        for key, (lo, hi) in expectations.items():
            try:
                obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
            except ClientError as e:
                code_str = e.response["Error"]["Code"]
                if code_str in ("NoSuchKey", "404", "NoSuchBucket"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"s3://{BUCKET_NAME}/{key} was not found. Upload the "
                            f"split before running the checkpoint."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))

            body = obj["Body"].read().decode("utf-8").splitlines()
            if not body:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{key} is empty.",
                )
            row_count = len(body)
            if not (lo <= row_count <= hi):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{key} has {row_count} rows, expected between {lo} and {hi}. "
                        f"Re-run the generator and re-upload."
                    ),
                )

            first_row = body[0].split(",")
            # Header check: first field of first row should be numeric (the label).
            try:
                label_val = int(first_row[0])
            except ValueError:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{key} first row first field is {first_row[0]!r}, not an "
                        f"integer label. The XGBoost built-in CSV format requires no "
                        f"header row and the label in column 0."
                    ),
                )
            if label_val not in (0, 1):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{key} first row label is {label_val}, expected 0 or 1. The "
                        f"generator emits a binary target."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Five blanks: bucket create, three uploads, role create, inline policy attach. The checkpoint message tells you whether a split is missing, the row count is off, or the first row looks like a header instead of a labeled row.

</details>

<details><summary>Hint 2 (direction)</summary>

In `us-east-1` use plain `s3.create_bucket(Bucket=BUCKET_NAME)` with no `CreateBucketConfiguration`. Each split uploads with `s3.put_object(Bucket=BUCKET_NAME, Key="input/<split>.csv", Body=<bytes>)`. The role is created with `iam.create_role(RoleName=..., AssumeRolePolicyDocument=json.dumps(trust_policy), Tags=[...])`. The inline policy attaches with `iam.put_role_policy(RoleName=..., PolicyName=..., PolicyDocument=json.dumps(inline_policy))`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the lines you need to fill in):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

s3.put_object(Bucket=BUCKET_NAME, Key="input/train.csv", Body=train_bytes)
s3.put_object(Bucket=BUCKET_NAME, Key="input/validation.csv", Body=val_bytes)
s3.put_object(Bucket=BUCKET_NAME, Key="input/test.csv", Body=test_bytes)

iam.create_role(
    RoleName=TRAINING_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="labrun mla lab 05 training role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.put_role_policy(
    RoleName=TRAINING_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)
```

</details>

**Wallet check.** One S3 bucket, three CSV uploads at roughly 200 KB combined, an IAM role, and an inline policy. S3 control-plane plus three put_object calls sits inside the always-free tier. IAM is free. Damage so far: about $0.00. The HPO sweeps are the line items that show up next.

## Task 2: Run the baseline XGBoost training job and capture validation AUC

The baseline is the no-tuning anchor. One training job with a single set of reasonable hyperparameters, fitting on train and reporting validation AUC. The baseline AUC is the number the two HPO sweeps will try to beat.

Build it in this order:

1. Submit a `sm.create_training_job` call named `labrun-xgboost-built-in-training-and-hpo-baseline` against the XGBoost built-in container.
2. Set `objective="binary:logistic"`, `eval_metric="auc"`, `num_round=50`, `max_depth=5`, `eta=0.2`, `subsample=0.8`, `min_child_weight=1`.
3. Use `instance_type="ml.m5.xlarge"`, `instance_count=1`, `volume_size_in_gb=10`.
4. Wait for `TrainingJobStatus=Completed` (roughly 5 minutes).
5. Walk `FinalMetricDataList` for the `validation:auc` entry and capture the value into `BASELINE_AUC`.

The lab does not deploy this model. The baseline AUC is the receipt; the HPO sweeps in Tasks 3 and 4 are what gets compared. The reflection MCQ uses the baseline-vs-HPO delta as context.

In [ ]:
# NBVAL_SKIP
# Task 2: Baseline XGBoost training. Code phase plus observe phase: the cell
# kicks off the job and polls until Completed, then walks FinalMetricDataList
# for the validation:auc value.

def submit_baseline_training():
    sm.create_training_job(
        TrainingJobName=BASELINE_JOB,
        AlgorithmSpecification={
            "TrainingImage": XGBOOST_IMAGE_URI,
            "TrainingInputMode": "File",
        },
        RoleArn=TRAINING_ROLE_ARN,
        InputDataConfig=[
            {
                "ChannelName": "train",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType": "S3Prefix",
                        "S3Uri": f"s3://{BUCKET_NAME}/input/train.csv",
                        "S3DataDistributionType": "FullyReplicated",
                    }
                },
                "ContentType": "text/csv",
            },
            {
                "ChannelName": "validation",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType": "S3Prefix",
                        "S3Uri": f"s3://{BUCKET_NAME}/input/validation.csv",
                        "S3DataDistributionType": "FullyReplicated",
                    }
                },
                "ContentType": "text/csv",
            },
        ],
        OutputDataConfig={"S3OutputPath": f"s3://{BUCKET_NAME}/output/baseline/"},
        ResourceConfig={
            "InstanceType": "ml.m5.xlarge",
            "InstanceCount": 1,
            "VolumeSizeInGB": 10,
        },
        StoppingCondition={"MaxRuntimeInSeconds": 900},
        HyperParameters={
            "objective": "binary:logistic",
            "num_round": "50",
            "max_depth": "5",
            "eta": "0.2",
            "subsample": "0.8",
            "min_child_weight": "1",
            "eval_metric": "auc",
        },
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )


# YOUR CODE: Submit the baseline training job by calling submit_baseline_training().
print(f"Submitted baseline training job: {BASELINE_JOB}")

print("XGBoost is fitting the baseline, give it about 5 minutes...")
elapsed = 0
status = "InProgress"
while elapsed < 900:
    resp = sm.describe_training_job(TrainingJobName=BASELINE_JOB)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Baseline reached status {status}. Inspect the job in the SageMaker console.")
    raise SystemExit(1)

resp = sm.describe_training_job(TrainingJobName=BASELINE_JOB)
final_metrics = resp.get("FinalMetricDataList", [])
for metric in final_metrics:
    if metric["MetricName"] == "validation:auc":
        BASELINE_AUC = metric["Value"]
        break

print(f"Baseline {BASELINE_JOB} completed in roughly {elapsed}s.")
print(f"Baseline validation:auc = {BASELINE_AUC}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Baseline training job is Completed with a captured validation:auc
# between 0 and 1, ideally above 0.65 for a working XGBoost run on this dataset.

def checkpoint_2(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_training_job(TrainingJobName=BASELINE_JOB)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Training job {BASELINE_JOB} was not found. Submit the baseline "
                        f"before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["TrainingJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline {BASELINE_JOB} status is {job['TrainingJobStatus']}, "
                    f"not Completed. Wait for the poll loop in the task cell to finish."
                ),
            )

        final_metrics = job.get("FinalMetricDataList", [])
        val_auc = None
        for metric in final_metrics:
            if metric["MetricName"] == "validation:auc":
                val_auc = metric["Value"]
                break
        if val_auc is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline {BASELINE_JOB} has no validation:auc in FinalMetricDataList. "
                    f"Confirm HyperParameters['eval_metric'] is 'auc' and the validation "
                    f"channel was wired correctly."
                ),
            )
        if not (0.0 <= val_auc <= 1.0):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline validation:auc = {val_auc}, outside the [0, 1] range. "
                    f"Something is wrong with the validation split or the metric name."
                ),
            )
        if val_auc < 0.6:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline validation:auc = {val_auc:.3f}, below 0.6. The dataset is "
                    f"deterministic and a sensible XGBoost baseline should land around "
                    f"0.7-0.8. Re-check the train/validation split sizes."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

One blank: call `submit_baseline_training()`. The helper does the full `create_training_job` call with the right hyperparameters. The checkpoint message tells you whether the job exists, is still running, or has a malformed validation AUC.

</details>

<details><summary>Hint 2 (direction)</summary>

`submit_baseline_training()` takes no arguments. It uses `ml.m5.xlarge`, `num_round=50`, and `eval_metric="auc"`. The poll loop waits up to 15 minutes; the typical wall-clock is roughly 5 minutes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the one line you need to fill in):

```python
submit_baseline_training()
```

</details>

**Wallet check.** One XGBoost training job on ml.m5.xlarge ($0.230 per hour) for roughly 5 minutes lands at about two cents. The XGBoost container image is AWS-published and free. Damage so far: about $0.02. The HPO sweeps coming up are 20 more trials at the same per-trial cost.

## Task 3: Architecture A - run the Bayesian HPO sweep (10 trials, 2 parallel)

SageMaker Automatic Model Tuning ships four strategies: Random, Bayesian, Hyperband, Grid. Bayesian builds a probabilistic surrogate of the objective as trials complete, then samples the next trial from the surrogate's posterior. With 10 trials and 2 in parallel, Bayesian sees enough completed trials by trial 5 or 6 to focus the search on promising regions.

Build it in this order:

1. Construct the HPO config: four hyperparameter ranges (`max_depth` IntegerParameter 3-10, `eta` ContinuousParameter 0.01-0.3, `subsample` ContinuousParameter 0.5-1.0, `min_child_weight` IntegerParameter 1-10).
2. Set objective to `MetricName="validation:auc"`, `Type="Maximize"`.
3. Set `Strategy="Bayesian"`, `MaxNumberOfTrainingJobs=10`, `MaxParallelTrainingJobs=2`, `EarlyStoppingType="Off"`. Early stopping off keeps the budget comparison honest with Random.
4. Submit via `sm.create_hyper_parameter_tuning_job` against the same XGBoost container, same instance type (`ml.m5.xlarge`), same train and validation channels as the baseline.
5. Poll until `HyperParameterTuningJobStatus=Completed`.
6. Capture `BestTrainingJob.FinalHyperParameterTuningJobObjectiveMetric.Value` into `BAYES_BEST`. Walk `list_training_jobs_for_hyper_parameter_tuning_job` sorted by `TrainingStartTime` to find the trial index where that best landed; record it into `BAYES_STEPS_TO_BEST`.

`MaxParallelTrainingJobs=2` is the deliberate choice. The brief and the reflection MCQ both surface why: cranking parallelism up degrades Bayesian's information-per-trial. The lab keeps the parallelism honest.

In [ ]:
# NBVAL_SKIP
# Task 3: Architecture A - Bayesian HPO sweep. Code phase plus observe phase:
# the cell kicks off the tuning job and polls until Completed, then captures
# the best objective and the steps-to-best trial index.


def build_static_hyperparameters() -> dict:
    return {
        "objective": "binary:logistic",
        "num_round": "50",
        "eval_metric": "auc",
    }


def build_tuning_config(strategy: str) -> dict:
    return {
        "Strategy": strategy,
        "HyperParameterTuningJobObjective": {
            "MetricName": "validation:auc",
            "Type": "Maximize",
        },
        "ResourceLimits": {
            "MaxNumberOfTrainingJobs": 10,
            "MaxParallelTrainingJobs": 2,
        },
        "ParameterRanges": {
            "IntegerParameterRanges": [
                {"Name": "max_depth", "MinValue": "3", "MaxValue": "10"},
                {"Name": "min_child_weight", "MinValue": "1", "MaxValue": "10"},
            ],
            "ContinuousParameterRanges": [
                {"Name": "eta", "MinValue": "0.01", "MaxValue": "0.3"},
                {"Name": "subsample", "MinValue": "0.5", "MaxValue": "1.0"},
            ],
        },
        "TrainingJobEarlyStoppingType": "Off",
    }


def build_training_job_definition() -> dict:
    return {
        "StaticHyperParameters": build_static_hyperparameters(),
        "AlgorithmSpecification": {
            "TrainingImage": XGBOOST_IMAGE_URI,
            "TrainingInputMode": "File",
            "MetricDefinitions": [
                {"Name": "validation:auc", "Regex": "validation-auc:([0-9\\.]+)"},
            ],
        },
        "RoleArn": TRAINING_ROLE_ARN,
        "InputDataConfig": [
            {
                "ChannelName": "train",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType": "S3Prefix",
                        "S3Uri": f"s3://{BUCKET_NAME}/input/train.csv",
                        "S3DataDistributionType": "FullyReplicated",
                    }
                },
                "ContentType": "text/csv",
            },
            {
                "ChannelName": "validation",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType": "S3Prefix",
                        "S3Uri": f"s3://{BUCKET_NAME}/input/validation.csv",
                        "S3DataDistributionType": "FullyReplicated",
                    }
                },
                "ContentType": "text/csv",
            },
        ],
        "OutputDataConfig": {"S3OutputPath": f"s3://{BUCKET_NAME}/output/{{strategy}}/"},
        "ResourceConfig": {
            "InstanceType": "ml.m5.xlarge",
            "InstanceCount": 1,
            "VolumeSizeInGB": 10,
        },
        "StoppingCondition": {"MaxRuntimeInSeconds": 900},
    }


def submit_tuning_job(job_name: str, strategy: str) -> None:
    job_def = build_training_job_definition()
    # Replace the placeholder output prefix with the strategy-specific path.
    job_def["OutputDataConfig"]["S3OutputPath"] = (
        f"s3://{BUCKET_NAME}/output/tuning-{strategy.lower()}/"
    )
    sm.create_hyper_parameter_tuning_job(
        HyperParameterTuningJobName=job_name,
        HyperParameterTuningJobConfig=build_tuning_config(strategy),
        TrainingJobDefinition=job_def,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )


# YOUR CODE: Submit the Bayesian tuning job by calling
# submit_tuning_job(BAYES_JOB, "Bayesian").
print(f"Submitted Bayesian tuning job: {BAYES_JOB}")

print("Bayesian is exploring the search space, give it about 25 minutes...")
print("(Bayesian narrows the search as trials complete; each trial uses prior trials.)")
elapsed = 0
status = "InProgress"
while elapsed < 2400:  # 40-minute ceiling for the 10-trial sweep at 2 parallel
    resp = sm.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=BAYES_JOB)
    status = resp["HyperParameterTuningJobStatus"]
    counters = resp.get("TrainingJobStatusCounters", {})
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(
            f"  {elapsed}s elapsed, status: {status}, completed trials: "
            f"{counters.get('Completed', 0)}/10"
        )

if status != "Completed":
    print(f"Bayesian sweep ended with status {status}. Inspect the SageMaker console.")
    raise SystemExit(1)

best_job = resp.get("BestTrainingJob", {})
BAYES_BEST = best_job.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value")
best_job_name = best_job.get("TrainingJobName")

# Walk the per-trial list, sorted by start time, to find where the best landed.
trials = sm.list_training_jobs_for_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=BAYES_JOB,
    MaxResults=50,
)["TrainingJobSummaries"]
trials_sorted = sorted(trials, key=lambda t: t.get("TrainingStartTime", t.get("CreationTime")))
for i, t in enumerate(trials_sorted, start=1):
    if t.get("TrainingJobName") == best_job_name:
        BAYES_STEPS_TO_BEST = i
        break

print(f"Bayesian sweep {BAYES_JOB} completed in roughly {elapsed}s.")
print(f"  Best validation:auc: {BAYES_BEST}")
print(f"  Best trial: {best_job_name} (trial {BAYES_STEPS_TO_BEST} of 10)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Bayesian tuning job is Completed, ran with Strategy=Bayesian,
# captured a best objective.

def checkpoint_3(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_hyper_parameter_tuning_job(
                HyperParameterTuningJobName=BAYES_JOB
            )
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Tuning job {BAYES_JOB} was not found. Submit the Bayesian "
                        f"sweep before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["HyperParameterTuningJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bayesian tuning job {BAYES_JOB} status is "
                    f"{job['HyperParameterTuningJobStatus']!r}, not Completed. Wait for "
                    f"the poll loop to finish."
                ),
            )

        strategy = job.get("HyperParameterTuningJobConfig", {}).get("Strategy")
        if strategy != "Bayesian":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Tuning job {BAYES_JOB} ran with Strategy={strategy!r}, expected "
                    f"'Bayesian'. Recreate with Strategy='Bayesian'."
                ),
            )

        counters = job.get("TrainingJobStatusCounters", {})
        if counters.get("Completed", 0) < 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bayesian sweep reported {counters.get('Completed', 0)} completed "
                    f"trials, expected 10. Check the SageMaker console for failed trials."
                ),
            )

        best = job.get("BestTrainingJob", {})
        best_val = best.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value")
        if best_val is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bayesian sweep has no captured best objective. Confirm the "
                    f"TrainingJobObjective MetricName is 'validation:auc'."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

One blank: call `submit_tuning_job(BAYES_JOB, "Bayesian")`. The helper builds the full HPO config with the right ranges, the right strategy, and the right budget. The poll loop and the best-objective capture are already written.

</details>

<details><summary>Hint 2 (direction)</summary>

`submit_tuning_job` takes the job name and the strategy string. It uses `MaxNumberOfTrainingJobs=10`, `MaxParallelTrainingJobs=2`, `EarlyStoppingType="Off"`, and the four hyperparameter ranges. The sweep runs roughly 25 minutes wall-clock with 2 trials in parallel.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the one line you need to fill in):

```python
submit_tuning_job(BAYES_JOB, "Bayesian")
```

</details>

**Wallet check.** Ten XGBoost trials on ml.m5.xlarge at $0.230 per hour for roughly 5 minutes each lands at about four cents (10 × 5/60 × 0.230). The two-parallel wall-clock is roughly 25 minutes; the cost is the same regardless of parallelism. Damage so far: about $0.06. Bayesian and Random both have the same compute bill; the comparison is on convergence, not on dollars.

## Task 4: Architecture B - run the Random HPO sweep (same budget, same ranges)

Random search samples hyperparameters uniformly from the specified ranges. It does not use information from completed trials. With 10 trials, Random has the same compute budget as Bayesian and the same ranges; the only thing that changes is which 10 configurations get sampled.

Build it in this order:

1. Submit the second tuning job with the same configuration EXCEPT `Strategy="Random"`.
2. Poll until Completed.
3. Capture `RANDOM_BEST` and `RANDOM_STEPS_TO_BEST` the same way Task 3 did for Bayesian.

The reflection MCQ in this lab asks when Random wins, when Bayesian wins, and when Hyperband becomes the right pick instead. The lab's job is to give the student a real result on a real dataset, not to declare a winner.

In [ ]:
# NBVAL_SKIP
# Task 4: Architecture B - Random HPO sweep. Same code path, different strategy.

# YOUR CODE: Submit the Random tuning job by calling
# submit_tuning_job(RANDOM_JOB, "Random").
print(f"Submitted Random tuning job: {RANDOM_JOB}")

print("Random is sampling the search space uniformly, give it about 25 minutes...")
print("(Random does not learn across trials; each trial is independent.)")
elapsed = 0
status = "InProgress"
while elapsed < 2400:
    resp = sm.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=RANDOM_JOB)
    status = resp["HyperParameterTuningJobStatus"]
    counters = resp.get("TrainingJobStatusCounters", {})
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(
            f"  {elapsed}s elapsed, status: {status}, completed trials: "
            f"{counters.get('Completed', 0)}/10"
        )

if status != "Completed":
    print(f"Random sweep ended with status {status}. Inspect the SageMaker console.")
    raise SystemExit(1)

best_job = resp.get("BestTrainingJob", {})
RANDOM_BEST = best_job.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value")
best_job_name = best_job.get("TrainingJobName")

trials = sm.list_training_jobs_for_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=RANDOM_JOB,
    MaxResults=50,
)["TrainingJobSummaries"]
trials_sorted = sorted(trials, key=lambda t: t.get("TrainingStartTime", t.get("CreationTime")))
for i, t in enumerate(trials_sorted, start=1):
    if t.get("TrainingJobName") == best_job_name:
        RANDOM_STEPS_TO_BEST = i
        break

print(f"Random sweep {RANDOM_JOB} completed in roughly {elapsed}s.")
print(f"  Best validation:auc: {RANDOM_BEST}")
print(f"  Best trial: {best_job_name} (trial {RANDOM_STEPS_TO_BEST} of 10)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Random tuning job is Completed, ran with Strategy=Random,
# captured a best objective.

def checkpoint_4(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_hyper_parameter_tuning_job(
                HyperParameterTuningJobName=RANDOM_JOB
            )
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Tuning job {RANDOM_JOB} was not found. Submit the Random "
                        f"sweep before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["HyperParameterTuningJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Random tuning job {RANDOM_JOB} status is "
                    f"{job['HyperParameterTuningJobStatus']!r}, not Completed. Wait for "
                    f"the poll loop to finish."
                ),
            )

        strategy = job.get("HyperParameterTuningJobConfig", {}).get("Strategy")
        if strategy != "Random":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Tuning job {RANDOM_JOB} ran with Strategy={strategy!r}, expected "
                    f"'Random'. Recreate with Strategy='Random'."
                ),
            )

        counters = job.get("TrainingJobStatusCounters", {})
        if counters.get("Completed", 0) < 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Random sweep reported {counters.get('Completed', 0)} completed "
                    f"trials, expected 10. Check the SageMaker console for failed trials."
                ),
            )

        best = job.get("BestTrainingJob", {})
        best_val = best.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value")
        if best_val is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Random sweep has no captured best objective."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

One blank: call `submit_tuning_job(RANDOM_JOB, "Random")`. The helper builds the same config as Bayesian; the only field that changes is the strategy string.

</details>

<details><summary>Hint 2 (direction)</summary>

The two sweeps must share the same `MaxNumberOfTrainingJobs`, `MaxParallelTrainingJobs`, hyperparameter ranges, and objective. `submit_tuning_job` already enforces that by sharing the helpers `build_tuning_config(strategy)` and `build_training_job_definition()`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the one line you need to fill in):

```python
submit_tuning_job(RANDOM_JOB, "Random")
```

</details>

**Wallet check.** Another 10 trials at the same per-trial cost. Damage so far: about $0.10. The math: 1 baseline + 20 HPO trials × 5 minutes × ml.m5.xlarge at $0.230 per hour = roughly 10 cents. Your morning coffee still wins this fight.

## Task 5: Build the side-by-side summary and surface the comparison metric

The two sweeps are done. Both produced a best objective and a steps-to-best trial index. Build the summary table the team's lead engineer wants:

- Baseline validation AUC vs Bayesian best AUC vs Random best AUC.
- Bayesian steps-to-best vs Random steps-to-best (lower means the strategy found its best earlier in the budget).
- Per-trial AUC trajectories: at each trial index 1-10, the running best-so-far for each strategy.

Checkpoint 5 captures these into a `comparisonMetric` string that the Option D Colab card surfaces on PASS. The validator confirms all four numbers (`bayes_best`, `bayes_steps_to_best`, `random_best`, `random_steps_to_best`) are non-zero and finite, and re-queries `list_training_jobs_for_hyper_parameter_tuning_job` to confirm the per-trial trajectories actually exist in the AWS side.

Per blueprint Section 21, the validator does NOT make a Bayesian-vs-Random superiority assertion in code. The result on a 10-trial budget and a small search space can land within noise. The comparative call is the student's, made in the reflection cell.

In [ ]:
# NBVAL_SKIP
# Task 5: Build the side-by-side summary. No new AWS calls; the comparison
# is a print-out for the student to read alongside the captured metrics.


def trial_trajectory(tuning_job_name: str) -> list:
    trials = sm.list_training_jobs_for_hyper_parameter_tuning_job(
        HyperParameterTuningJobName=tuning_job_name,
        MaxResults=50,
    )["TrainingJobSummaries"]
    trials_sorted = sorted(
        trials, key=lambda t: t.get("TrainingStartTime", t.get("CreationTime"))
    )
    best_so_far = 0.0
    trajectory = []
    for t in trials_sorted:
        val = t.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value", 0.0)
        if val and val > best_so_far:
            best_so_far = val
        trajectory.append(best_so_far)
    return trajectory


bayes_trajectory = trial_trajectory(BAYES_JOB)
random_trajectory = trial_trajectory(RANDOM_JOB)

print("Per-trial best-so-far validation AUC:")
print(f"  trial:   {'  '.join(f'{i:>5}' for i in range(1, 11))}")
print(
    f"  bayes:   {'  '.join(f'{v:.3f}' for v in bayes_trajectory[:10]):<60}"
)
print(
    f"  random:  {'  '.join(f'{v:.3f}' for v in random_trajectory[:10]):<60}"
)
print()
print(f"Baseline validation:auc: {BASELINE_AUC}")
print(f"Bayesian best validation:auc: {BAYES_BEST} (best at trial {BAYES_STEPS_TO_BEST} of 10)")
print(f"Random best validation:auc: {RANDOM_BEST} (best at trial {RANDOM_STEPS_TO_BEST} of 10)")
print()
print("Receipt for the lead engineer:")
print(f"  Bayesian beat baseline by {(BAYES_BEST or 0) - (BASELINE_AUC or 0):+.3f} AUC points.")
print(f"  Random beat baseline by {(RANDOM_BEST or 0) - (BASELINE_AUC or 0):+.3f} AUC points.")
print(f"  Bayesian vs Random: {(BAYES_BEST or 0) - (RANDOM_BEST or 0):+.3f} AUC points.")
print(
    f"  Steps-to-best: Bayesian {BAYES_STEPS_TO_BEST}, Random {RANDOM_STEPS_TO_BEST}."
)

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: All four numbers (bayes_best, bayes_steps_to_best, random_best,
# random_steps_to_best) are captured and well-formed. The validator also
# re-queries the two tuning jobs to confirm the trial summaries are present.
# Per blueprint Section 21, the validator does NOT compute a Bayesian-vs-Random
# winner; the comparative call lives in the reflection cell.

def checkpoint_5(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        for label, value in [
            ("BAYES_BEST", BAYES_BEST),
            ("BAYES_STEPS_TO_BEST", BAYES_STEPS_TO_BEST),
            ("RANDOM_BEST", RANDOM_BEST),
            ("RANDOM_STEPS_TO_BEST", RANDOM_STEPS_TO_BEST),
        ]:
            if value is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{label} is None. Run the summary cell after both HPO sweeps "
                        f"complete so the comparison values are captured."
                    ),
                )

        if not (0.0 < BAYES_BEST < 1.0):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"BAYES_BEST = {BAYES_BEST}, outside the (0, 1) AUC range. "
                    f"Recheck the captured value from BestTrainingJob."
                ),
            )
        if not (0.0 < RANDOM_BEST < 1.0):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"RANDOM_BEST = {RANDOM_BEST}, outside the (0, 1) AUC range."
                ),
            )
        if not (1 <= BAYES_STEPS_TO_BEST <= 10):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"BAYES_STEPS_TO_BEST = {BAYES_STEPS_TO_BEST}, outside the [1, 10] "
                    f"trial-index range."
                ),
            )
        if not (1 <= RANDOM_STEPS_TO_BEST <= 10):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"RANDOM_STEPS_TO_BEST = {RANDOM_STEPS_TO_BEST}, outside the [1, 10] "
                    f"trial-index range."
                ),
            )

        # Re-query the AWS side to confirm both tuning jobs hold trial summaries.
        for job_name in (BAYES_JOB, RANDOM_JOB):
            trials = sm_client.list_training_jobs_for_hyper_parameter_tuning_job(
                HyperParameterTuningJobName=job_name,
                MaxResults=50,
            )["TrainingJobSummaries"]
            if len(trials) < 10:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{job_name} reports only {len(trials)} training-job summaries. "
                        f"The sweep did not finish all 10 trials."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Zero blanks. The summary cell runs entirely from the captured `BAYES_BEST`, `BAYES_STEPS_TO_BEST`, `RANDOM_BEST`, `RANDOM_STEPS_TO_BEST`, and the AWS trial summaries. If the checkpoint fails, one of the four values is None or out of range, which means an earlier task did not capture its result.

</details>

<details><summary>Hint 2 (direction)</summary>

If `BAYES_BEST` or `BAYES_STEPS_TO_BEST` is None, re-run Task 3's poll loop and confirm `BestTrainingJob` came back from `describe_hyper_parameter_tuning_job`. Same for Random in Task 4. The summary cell does not write to AWS; it only reads and prints.

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is nothing to fill in. The summary cell prints the per-trial trajectories and the side-by-side numbers. If the checkpoint fails, the fix is in Task 3 or Task 4, not here.

</details>

**Wallet check.** No new compute in this cell. The two `list_training_jobs_for_hyper_parameter_tuning_job` calls are free. Damage so far: still about $0.10. The receipt for the lead engineer is the cheapest part of the lab.

## Cleanup

The two HPO tuning jobs and the baseline training job cannot be deleted via API. Completed jobs auto-expire and bill nothing once Completed. The cleanup cell defensively calls `stop_training_job` and `stop_hyper_parameter_tuning_job` on any in-flight job in case the kernel was restarted mid-sweep. Then `run_cleanup` walks the IAM role and S3 bucket.

Re-running the cell is safe. Each manual stop is wrapped in try/except that swallows "already gone" errors.

In [ ]:
# NBVAL_SKIP
# Cleanup. Defensive stop on any in-flight job (training or tuning), then
# run_cleanup walks iam_role + s3_bucket.

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []

# Stop any in-flight training jobs.
for job_name in (BASELINE_JOB,):
    try:
        resp = sm_cleanup.describe_training_job(TrainingJobName=job_name)
        if resp["TrainingJobStatus"] == "InProgress":
            try:
                sm_cleanup.stop_training_job(TrainingJobName=job_name)
                print(f"Requested stop on in-flight training job {job_name}")
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ValidationException",
                    "ResourceNotFound",
                ):
                    manual_warnings.append(
                        f"FAILED TO STOP: training_job {job_name!r}. Error: {e}."
                    )
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
            manual_warnings.append(
                f"FAILED TO DESCRIBE: training_job {job_name!r}. Error: {e}."
            )

# Stop any in-flight tuning jobs.
for job_name in (BAYES_JOB, RANDOM_JOB):
    try:
        resp = sm_cleanup.describe_hyper_parameter_tuning_job(
            HyperParameterTuningJobName=job_name
        )
        if resp["HyperParameterTuningJobStatus"] == "InProgress":
            try:
                sm_cleanup.stop_hyper_parameter_tuning_job(
                    HyperParameterTuningJobName=job_name
                )
                print(f"Requested stop on in-flight tuning job {job_name}")
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ValidationException",
                    "ResourceNotFound",
                ):
                    manual_warnings.append(
                        f"FAILED TO STOP: tuning_job {job_name!r}. Error: {e}."
                    )
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
            manual_warnings.append(
                f"FAILED TO DESCRIBE: tuning_job {job_name!r}. Error: {e}."
            )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
critical_destroyed = 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean") else "dirty"
cleanup(status=final_status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.08 to $0.15.** One baseline training job plus 20 HPO trials on ml.m5.xlarge at $0.230 per hour for roughly 5 minutes per trial. The math: 1 × 5/60 × 0.230 + 20 × 5/60 × 0.230 = about ten cents. The XGBoost container image is AWS-published and free. S3 storage for the three splits and the per-trial model artifacts is fractions of a penny. IAM and STS are free. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the four HPO strategies SageMaker Automatic Model Tuning supports (Random, Bayesian, Hyperband, Grid). For each one, write one sentence on how it picks the next set of hyperparameters and one sentence on when you would pick it. What is the role of `MaxParallelTrainingJobs` in Bayesian convergence, and why does cranking it up actually hurt you? Why did this lab pin parallelism at 2 instead of 10?

2. The lab compared Bayesian and Random on a fixed 10-trial budget across 4 hyperparameters. Walk through what changes if the budget jumps to 200 trials and the search space jumps to 12 hyperparameters. Which strategy gets more relative benefit, and is there a point where Hyperband becomes the right pick instead of either Bayesian or Random? What changes again if the trials are deep-learning runs where early-stop signals correlate strongly with full-run performance?

## Exam-Style Practice

**Question 1 (Domain 2, HPO strategy selection; constraint-driven comparison per blueprint Section 21):**

A team has a fixed compute budget of 200 trials for HPO on an XGBoost model with 8 hyperparameters. The training-set size is large enough that early-stopped trials reliably correlate with full-run performance. The team wants to maximize the chance of finding the best configuration within budget. Which strategy fits?

A. Random search. Sampling uniformly across the search space avoids local optima at large budgets.

B. Bayesian search. The probabilistic surrogate uses information from prior trials to focus on promising regions.

C. Hyperband. Bandit-style early stopping prunes underperforming configurations and concentrates compute on the survivors.

D. Grid search. Exhaustive coverage at fixed step sizes guarantees the best configuration is in the grid.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong on "maximize chance of best": Random does not learn across trials; it spends compute uniformly across the search space and wastes budget on obviously-bad configurations.
- B is partially right: Bayesian uses information from prior trials but cannot early-stop trials that are running. With 200 trials and 8 hyperparameters Bayesian works well, but Hyperband typically converges faster when early-stop signals correlate with full-run performance, which is the explicit scenario condition.
- C is correct: Hyperband uses bandit-style successive halving to prune trials that perform poorly in early epochs and concentrate budget on the most promising configurations. The scenario conditions (early-stopped trials reliably correlate with full-run performance and fixed compute budget of 200 trials) are the exact conditions where Hyperband wins.
- D is wrong on 8 hyperparameters: a grid that covers 8 hyperparameters at any reasonable step size explodes well past 200 trials. Grid is the right pick only at very low dimensionality with few discrete values.

</details>

**Question 2 (Domain 2, Bayesian HPO and parallelism):**

A team runs a Bayesian HPO sweep with `MaxNumberOfTrainingJobs=50` and `MaxParallelTrainingJobs=10`. Convergence is worse than the team's prior Bayesian sweep that used `MaxNumberOfTrainingJobs=50` and `MaxParallelTrainingJobs=2`. Same dataset, same hyperparameter ranges. Why?

A. SageMaker Automatic Model Tuning does not support Bayesian with parallelism above 4; the runs at parallelism 10 silently fell back to Random.

B. Bayesian picks the next trial using information from completed trials; running 10 trials in parallel means each batch of 10 is picked using stale information from before the batch started.

C. The ml.m5.xlarge instance type the team used does not support parallel Bayesian; they should switch to ml.c5.4xlarge.

D. Bayesian search requires MaxParallelTrainingJobs=1 to function correctly; AWS warns about this in the HPO docs.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: AWS does not silently fall back to Random for any parallelism level. The strategy is what the user requests.
- B is correct: Bayesian search builds a surrogate model of the objective as trials complete, then samples the next trial from the surrogate's posterior. At MaxParallelTrainingJobs=10, the next batch of 10 trials is chosen using whatever the surrogate looked like before the batch started. The more parallelism, the less Bayesian uses each trial's information; at infinite parallelism, Bayesian degenerates to Random.
- C is wrong: instance type does not affect HPO strategy convergence. Instance type affects per-trial speed, not the search algorithm's behavior.
- D is wrong: Bayesian works at any parallelism, but the asymptotic information per trial degrades as parallelism increases. There is no AWS hard requirement to use parallelism 1; the practical recommendation is 2-4 for most workloads.

</details>